## 1. Setup and Dependencies

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from Bio import Entrez, SeqIO
import time
from tqdm.auto import tqdm
import re
from transformers import BertModel, BertTokenizer
import torch

## 2. Load Phage Metadata (All Phages from PA.csv)

In [ ]:
df_pa = pd.read_csv("PA.csv")
phages_to_fetch = df_pa["ncbi_acc"].unique().tolist()
print(f"Phages to fetch from NCBI ({len(phages_to_fetch)} total):")
print(phages_to_fetch)

## 3. Fetch FASTA locally by Phage Name  
This cell loads pre-downloaded FASTA files from the local directory

In [ ]:
LOCAL_FASTA_DIR = Path("FASTA files")  # e.g., Path(r"FASTA files")

OUTPUT_DIR = Path("phage_data")
OUTPUT_DIR.mkdir(exist_ok=True)
GENOMES_DIR = OUTPUT_DIR / "genomes"
GENOMES_DIR.mkdir(exist_ok=True)

def _clean_name(phage_name: str) -> str:
    # mimic original: strip parenthetical content for matching
    return phage_name.split("(")[0].strip() if "(" in phage_name else phage_name.strip()

def _find_local_fasta(phage_name: str) -> Path | None:
    """
    Try hard to find a local FASTA for a given phage_name.
    Matching order:
      1) exact filename: {phage_name}.fasta/.fa/.fna (with and without parentheses stripped)
      2) filename contains cleaned name (case-insensitive)
    """
    name_raw = phage_name.strip()
    name_clean = _clean_name(phage_name)

    exts = [".fasta", ".fa", ".fna", ".fas"]
    # 1) exact matches
    for base in [name_raw, name_clean]:
        for ext in exts:
            p = LOCAL_FASTA_DIR / f"{base}{ext}"
            if p.exists():
                return p

    # 2) fuzzy contains match (case-insensitive)
    # WARNING: if you have multiple similar names, this can pick the wrong one.
    # If that happens, fix filenames to exact-match phage names in PA.csv.
    if LOCAL_FASTA_DIR.exists():
        candidates = []
        pattern = re.compile(re.escape(name_clean), re.IGNORECASE)
        for ext in exts:
            candidates.extend(LOCAL_FASTA_DIR.glob(f"*{ext}"))
        for p in sorted(candidates):
            if pattern.search(p.stem):
                return p

    return None

def load_phage_genome_from_local(phage_name: str):
    """
    Returns (phage_name, accession, sequence) or None.
    Mimics original behavior:
      - accession = seq_record.id.split(".")[0]
      - only accept sequences > 5000 bp
    """
    fasta_path = _find_local_fasta(phage_name)
    if fasta_path is None:
        return None

    try:
        seq_record = next(SeqIO.parse(str(fasta_path), "fasta"), None)
        if seq_record is None:
            return None

        seq = str(seq_record.seq)
        if len(seq) <= 1:
            return None  

        acc = seq_record.id.split(".")[0]
        return (phage_name, acc, seq)

    except Exception as e:
        print(f"  Error loading {phage_name} from {fasta_path.name}: {e}")
        return None

seqs = {}
for name in tqdm(phages_to_fetch, desc="Loading local FASTA"):
    result = load_phage_genome_from_local(name)
    if result:
        pname, acc, seq = result
        seqs[pname] = {"accession": acc, "sequence": seq}

        fasta_out = GENOMES_DIR / f"{pname}.fasta"
        with open(fasta_out, "w", encoding="utf-8") as f:
            f.write(f">{acc} {pname}\n{seq}\n")

        print(f"  {pname}: {acc}, len={len(seq)}")
    else:
        print(f"  {name}: not found")

print(f"\nFetched {len(seqs)} genomes")

## 4. Compute GC Content and Length

In [ ]:
def gc_content(seq: str) -> float:
    s = seq.upper().replace("N", "")
    if not s:
        return 0.0
    gc = s.count("G") + s.count("C")
    return 100.0 * gc / len(s)

rows = []
for name, data in seqs.items():
    seq = data["sequence"]
    rows.append({
        "phage": name,
        "ncbi_acc": data["accession"],
        "length": len(seq),
        "gc_content": gc_content(seq),
    })

df_meta = pd.DataFrame(rows)
print("Genome metadata:")
display(df_meta)

## 5. DNABERT Embedding

In [ ]:
tok = BertTokenizer.from_pretrained(
    "zhihan1996/DNA_bert_6",
    trust_remote_code=True
)

bert = BertModel.from_pretrained(
    "zhihan1996/DNA_bert_6",
    trust_remote_code=True
).eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
bert.to(device)

In [ ]:
def embed_sequence(
    seq: str,
    k: int = 6,
    window: int = 510,
    stride: int = 250,
    batch_windows: int = 16, 
) -> np.ndarray:
    seq = seq.upper().replace("N", "")
    windows = [
        " ".join(seq[i + j : i + j + k] for j in range(window))
        for i in range(0, len(seq) - k - window + 1, stride)
    ]
    if not windows:
        return np.zeros(768, dtype=np.float32)

    cls_sum = None
    n_total = 0

    for start in range(0, len(windows), batch_windows):
        batch = windows[start : start + batch_windows]

        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
        enc = {kk: vv.to(device) for kk, vv in enc.items()}

        with torch.no_grad():
            out = bert(**enc).last_hidden_state[:, 0, :]   # (batch, 768)

        batch_mean_sum = out.sum(dim=0)  # sum CLS across this batch
        cls_sum = batch_mean_sum if cls_sum is None else (cls_sum + batch_mean_sum)
        n_total += out.shape[0]

        # reduce peak memory pressure
        del enc, out
        if device == "cuda":
            torch.cuda.empty_cache()

    return (cls_sum / n_total).cpu().numpy()

embeddings = {}
for name, data in tqdm(seqs.items(), desc="Embedding"):
    try:
        vec = embed_sequence(data["sequence"])
        embeddings[name] = vec
    except Exception as e:
        print(f"  {name}: {e}")

emb_cols = [f"emb_{i}" for i in range(768)]
df_emb = pd.DataFrame([embeddings[n] for n in df_meta["phage"]], columns=emb_cols, index=df_meta["phage"])
df_emb = df_emb.reset_index()
df_full = df_meta.merge(df_emb, on="phage")
print(f"Embeddings shape: {df_full[emb_cols].shape}")

In [ ]:
df_full.to_csv('embeddings.csv')